In [1]:
import pandas as pd
from pathlib import Path
import pyarrow
assert pyarrow #import: enables pandas parquet support

#Variables
PICKUP, DROPOFF = "tpep_pickup_datetime", "tpep_dropoff_datetime"
COLUMNS = [PICKUP, DROPOFF, "trip_distance", "PULocationID", "DOLocationID"]
DISTANCE_THRESHOLD = 40
DURATION_THRESHOLD = 90
INVALID_ZONE_START = 264  # zones 264 and 265 are unknown/outside NYC

#Helper Functions
def read_columns(path):
    return pd.read_parquet(path, columns=COLUMNS)

def report(before, after):
    percent = after/before
    dropped_percent = 1 - percent
    preserved_percent = percent
    # noinspection string-format
    return (
        f"Previous Dataset Size: {before:,} \n"
        f"Current Dataset Size: {after:,} \n"
        f"Dropped: {before - after:,} lines \n"
        f"{dropped_percent :.2%} Dropped \n"
        f"{preserved_percent:.2%} Preserved"
    )

# This is need to make the files agnostic of the project root name as I don't know if we all have it as CodingChallenge
def find_project_root(marker="OriginalData_DoNotDelete", start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    raise FileNotFoundError(f"Could not find '{marker}' above {start}")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "OriginalData_DoNotDelete"
CLEANED_DIR = PROJECT_ROOT / "CleanedData"

# List of all files to process
FILE_NAMES = [
    "yellow_tripdata_2026-Jan",
    "yellow_tripdata_2026-Feb",
    "yellow_tripdata_2026-Mar",
    "yellow_tripdata_2026-Apr",
]

def clean_data(dataframe_original, verbose=True):
    start = len(dataframe_original)
    dataframeclean = dataframe_original.copy()

    # Calculate new features
    dataframeclean["duration_min"] = (dataframeclean[DROPOFF] - dataframeclean[PICKUP]).dt.total_seconds() / 60
    dataframeclean["pickup_hour"] = dataframeclean[PICKUP].dt.hour
    dataframeclean["pickup_dow"] = dataframeclean[PICKUP].dt.day_of_week # Fixed pandas warning

    steps = [
        (f"Duration > 0 & <= {DURATION_THRESHOLD}",
         lambda d: (d["duration_min"] > 0) & (d["duration_min"] <= DURATION_THRESHOLD)),
        (f"Distance in (0, {DISTANCE_THRESHOLD})]",
         lambda d: d["trip_distance"].between(0, DISTANCE_THRESHOLD, inclusive="right")),
        ("Drop zones 264/265",
         lambda d: (d["PULocationID"] < INVALID_ZONE_START) & (d["DOLocationID"] < INVALID_ZONE_START)),
    ]

    # Apply masks sequentially
    for name, mask in steps:
        before = len(dataframeclean)
        dataframeclean = dataframeclean.loc[mask(dataframeclean)] # Removed reset_index here
        if verbose:
            print(f"--- {name} ---\n{report(before, len(dataframeclean))}\n")

    if verbose:
        print(f"=== Overall ===\n{report(start, len(dataframeclean))}")

    return dataframeclean

In [2]:
all_cleaned_data = []
total_raw_rows = 0
total_clean_rows = 0

for file_name in FILE_NAMES:
    print(f"\n{'='*40}")
    print(f"Processing: {file_name}")
    print(f"{'='*40}")

    file_path = DATA_DIR / f"{file_name}.parquet"

    # 1. Load
    raw_df = read_columns(file_path)
    total_raw_rows += len(raw_df)

    # 2. Clean
    clean_df = clean_data(raw_df, verbose=True)
    total_clean_rows += len(clean_df)

    # 3. Store in list
    all_cleaned_data.append(clean_df)

# Combine all dataframes
print(f"\n{'='*40}")
print("Combining all months...")
print(f"{'='*40}")
dataframe_clean = pd.concat(all_cleaned_data, ignore_index=True)

print(f"\n=== FINAL TOTALS ===")
print(f"Raw Rows Across All Files: {total_raw_rows:,}")
print(f"Cleaned Rows Across All Files: {total_clean_rows:,}")
print(f"Overall Drop: {total_raw_rows - total_clean_rows:,} lines ({(1 - total_clean_rows/total_raw_rows):.2%})")


Processing: yellow_tripdata_2026-Jan
--- Duration > 0 & <= 90 ---
Previous Dataset Size: 3,724,889 
Current Dataset Size: 3,664,740 
Dropped: 60,149 lines 
1.61% Dropped 
98.39% Preserved

--- Distance in (0, 40)] ---
Previous Dataset Size: 3,664,740 
Current Dataset Size: 3,539,451 
Dropped: 125,289 lines 
3.42% Dropped 
96.58% Preserved

--- Drop zones 264/265 ---
Previous Dataset Size: 3,539,451 
Current Dataset Size: 3,517,183 
Dropped: 22,268 lines 
0.63% Dropped 
99.37% Preserved

=== Overall ===
Previous Dataset Size: 3,724,889 
Current Dataset Size: 3,517,183 
Dropped: 207,706 lines 
5.58% Dropped 
94.42% Preserved

Processing: yellow_tripdata_2026-Feb
--- Duration > 0 & <= 90 ---
Previous Dataset Size: 3,399,866 
Current Dataset Size: 3,345,383 
Dropped: 54,483 lines 
1.60% Dropped 
98.40% Preserved

--- Distance in (0, 40)] ---
Previous Dataset Size: 3,345,383 
Current Dataset Size: 3,222,303 
Dropped: 123,080 lines 
3.68% Dropped 
96.32% Preserved

--- Drop zones 264/265 --

In [3]:
# Save to new parquet file
path_to_data = CLEANED_DIR / "yellow_tripdata_2026_Combined_Cleaned.parquet"
path_to_data.parent.mkdir(parents=True, exist_ok=True)
dataframe_clean.to_parquet(path_to_data)
print(f"Saved combined cleaned data to {path_to_data}")

Saved combined cleaned data to D:\CodingChallenge\CleanedData\yellow_tripdata_2026_Combined_Cleaned.parquet


In [4]:
# Testing 
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error

dataframe_clean = pd.read_parquet(path_to_data)

FEATURES = [
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "pickup_hour",
    "pickup_dow",
]

numeric = ["trip_distance"]
categorical = ["PULocationID", "DOLocationID", "pickup_hour", "pickup_dow"]

X = dataframe_clean[FEATURES]
y = dataframe_clean["duration_min"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# sorts features into numerical or categorical data since some of the data isn't numerical e.g. zone id 250 isn't technically bigger than zone id 5
preprocess = ColumnTransformer([
    ("num", "passthrough", numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])

model = Pipeline([
    ("prep", preprocess),
    ("reg", LinearRegression()),
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# the typical size of prediction errors, with larger errors weighted more heavily
rmse = root_mean_squared_error(y_test, y_pred)

# the proportion of the variation in trip duration that the model explains
r2 = r2_score(y_test, y_pred)

print(f"Root Mean Squared Error: {rmse:.2f} minutes")
print(f"R-squared ($R^2$): {r2:.2f}")

Root Mean Squared Error: 7.23 minutes
R-squared ($R^2$): 0.69


It took 2m 6s for the overall

The data it gave:
Root Mean Squared Error: 7.23 minutes
R-squared ($R^2$): 0.69